Notebook for importing and cleaning the CSVs

In [30]:
#A block of code that loads vents and emotions csvs, and puts them into a dataframe

import pandas as pd
import matplotlib.pyplot as plt

vents = pd.read_csv("Vent/vents_metadata.csv.bz2")
emotions = pd.read_csv("Vent/emotions.csv")
categories = pd.read_csv("Vent/emotion_categories.csv")

df = pd.DataFrame(vents)

In [31]:
#merge vents and emotions together using the emotion ID
ventsMerg = vents.merge(emotions, left_on='emotion_id', right_on='id')
#merge with emotion categories
ventsMerg = ventsMerg.merge(categories, left_on='emotion_category_id', right_on="id")

print(ventsMerg.columns.tolist())

ventswithEmotions = ventsMerg[['id_x', 'user_id', 'name_x', 'name_y', 'reactions']]
ventswithEmotions = ventswithEmotions.rename(columns={'id_x': 'post_id', 'name_x': 'emotion', 'name_y': 'emotion_category'})

print(ventswithEmotions.head())

['emotion_id', 'user_id', 'created_at', 'reactions', 'id_x', 'emotion_category_id', 'name_x', 'enabled', 'id_y', 'name_y']
                                post_id                               user_id  \
0  a992d9f0-1b4c-4a6c-9d73-4cd7deb287ef  1a62fe90-d702-3051-b5fe-0a9c86adac56   
1  fe1ac197-3294-493f-ba9d-04c6bfbea10c  f05733e9-078c-3413-848d-a30a7f502ee9   
2  3180a95c-c03d-4a36-b78c-26d54d928049  336d37c1-9dfe-3454-a60d-dfa853f52bcc   
3  d84f9579-ba96-4818-93a5-7ff12f504098  f281696f-5be8-4b4c-bc44-056ebd6f4157   
4  abc354fa-7778-490b-81e9-01c6e7f776a0  f281696f-5be8-4b4c-bc44-056ebd6f4157   

       emotion   emotion_category  reactions  
0    Surprised           Surprise          0  
1     Stressed               Fear          6  
2          Sad            Sadness          0  
3     🐕 Good 🐕        🐶 Dog Day 🐶          1  
4  🐧 Melting 🐧  🌍 Earth Day '18 🌍        999  


Now the data is merged and we have post_ID, user_ID, emotion,reactions. 

**Next:**

1. Remove the 'junk' emotions (such as Halloween, World Cup, Vampire)
    1. Remove emojis from emotion tags (because many emotions are the same, plus or minus different emoji patterns)
    1. View all unique emotions
    1. Remove non-affective vents from the dataset

In [32]:
import re

def remove_emojis(text):
    return re.sub(r'[^\w\s,]', '', text).strip()

ventswithEmotions['emotion'] = ventswithEmotions['emotion'].apply(remove_emojis)

In [33]:
# 1.2 view all unique emotions
print(sorted(ventswithEmotions['emotion'].unique()))

['ANGELIC', 'APPRECIATIVE', 'Abandoned', 'Adaptable', 'Adoring', 'Adventurous', 'Affectionate', 'Afraid', 'Aging', 'Alienated', 'Alive', 'Allergic', 'Altruistic', 'Amazed', 'Amused', 'Anchored', 'Angelic', 'Angry', 'Annoyed', 'Anxious', 'Artistic', 'Ashamed', 'Astonished', 'Astronomical', 'Atorahble', 'Autumnal', 'Awake', 'Aware', 'Awkward', 'Azad', 'BLESSED', 'BOOKISH', 'Baited', 'Ballin', 'Bamboozled', 'Basking', 'Batty', 'Beachy', 'Believing', 'Berraco', 'Beyuletiful', 'Biconic', 'Bitten', 'Bitter', 'Blessed', 'Blinded', 'Blocked', 'Blooming', 'Bonded', 'BoneToParty', 'Booming', 'Bootiful', 'Bootyful', 'Bored', 'Brainy', 'Brave', 'Bready', 'Breaking', 'Breathtaking', 'Breezy', 'Brewtiful', 'Bright', 'Bromantic', 'Bubbly', 'Bundled up', 'Burnt', 'Busy', 'Buzzing', 'CHILLED', 'CLOSETED', 'COOL', 'CREEPY', 'Caffeinated', 'Calm', 'Caring', 'Catastrophic', 'Cautious', 'Celebrating', 'Celestial', 'Charitable', 'Charmed', 'Charming', 'Cheerful', 'Cheery', 'Cherished', 'Chill', 'Chilly', 'C

**I used Claude at this stage to select which emotions above were non affective and should be added to the delete lit**

In [34]:
junk_emotions = [
    # Seasonal/Event-based
    'Autumnal', 'Beachy', 'Breezy', 'Chilly', 'Cloudy', 'Rainy', 'Stormy', 
    'Slushy', 'Sunny', 'Springy', 'Toasty', 'Bundled up', 'Hibernating',
    
    # Holiday/Celebration specific
    'Festive', 'Ghoulish', 'Spooky', 'Spoopy', 'Creepy', 'Haunted', 'Eerie', 
    'Fangtastic', 'Grinchy', 'Jolly', 'Merry', 'Naughty', 'Nice', 'Trick or Treat', 
    'BoneToParty', 'Purranormal', 'Hissterical', 'Clawful',
    
    # Nonsense/Made-up words
    'Atorahble', 'Beyuletiful', 'Biconic', 'Bootyful', 'Bootiful', 'Brewtiful', 
    'Frosttea', 'Furocious', 'Greightful', 'Gwangbok', 'Jawesome', 'Merdeka', 
    'Mukt', 'Nezavisan', 'Pieceful', 'Skyiddish', 'Tropicool', 'Azad', 'Libre', 
    'Berraco', 'Einheitlich',
    
    # Activity-based
    'Gaming', 'Gossiping', 'Fasting', 'Feasting', 'Gathering', 'Huddling', 
    'Jammin', 'Partying', 'Polling', 'Praying', 'Rioting', 'Typing', 'Waving'
]

ventswithEmotions = ventswithEmotions[~ventswithEmotions['emotion'].isin(junk_emotions)]

print(f"Remaining emotions: {ventswithEmotions['emotion'].nunique()}")
print(f"Remaining rows: {len(ventswithEmotions)}")

Remaining emotions: 513
Remaining rows: 32853345


About 100 emotions deleted from the database, but only a small portion of total Vents removed

At this point the dataset contains emotions, and emotional category. Next is calculating and adding the 'follower count'. 

**Steps**

1. Load vent.edgelist (Column A follows Column B)
1. Count occurances/rows of a specific column A value 
    Read first unique value, count occurences, read next unique value.
1. Use the occurances counter as the follower count in our dataframe

In [35]:
followers = pd.read_csv("Vent/vent.edgelist/vent.edgelist", sep=" ", header=None, names=["user", "follower"])

follower_counts = followers['user'].value_counts().reset_index()
follower_counts.columns = ['user_id', 'follower_count']

ventswithEmotions = ventswithEmotions.merge(follower_counts, on='user_id', how='left')
print(ventswithEmotions.columns.tolist())
print(ventswithEmotions['user_id'].dtype, follower_counts['user_id'].dtype)

ventswithEmotions['emotion_category'] = ventswithEmotions['emotion_category'].apply(remove_emojis)
ventswithEmotions['follower_count'] = ventswithEmotions['follower_count'].fillna(0).astype(int)

print(ventswithEmotions.head())


['post_id', 'user_id', 'emotion', 'emotion_category', 'reactions', 'follower_count']
str str
                                post_id                               user_id  \
0  a992d9f0-1b4c-4a6c-9d73-4cd7deb287ef  1a62fe90-d702-3051-b5fe-0a9c86adac56   
1  fe1ac197-3294-493f-ba9d-04c6bfbea10c  f05733e9-078c-3413-848d-a30a7f502ee9   
2  3180a95c-c03d-4a36-b78c-26d54d928049  336d37c1-9dfe-3454-a60d-dfa853f52bcc   
3  d84f9579-ba96-4818-93a5-7ff12f504098  f281696f-5be8-4b4c-bc44-056ebd6f4157   
4  abc354fa-7778-490b-81e9-01c6e7f776a0  f281696f-5be8-4b4c-bc44-056ebd6f4157   

     emotion emotion_category  reactions  follower_count  
0  Surprised         Surprise          0               0  
1   Stressed             Fear          6               0  
2        Sad          Sadness          0               0  
3       Good          Dog Day          1             198  
4    Melting     Earth Day 18        999             198  


In [36]:
ventswithEmotions.to_csv("ventsCleaned")